# Machine Learning Foundation: MLOps, Training Pipelines & Neural Network Projects

## Course Overview

This notebook covers the essential MLOps practices for managing deep learning training workflows at scale. We'll explore:

1. **Deep Learning Training Pipelines** - Managing complex workflows with configuration management
2. **Compute Resource Management** - Cloud GPU optimization and cost control
3. **Experiment Tracking & Reproducibility** - MLflow, TensorBoard, and logging
4. **Neural Network Project Development** - End-to-end implementation with production practices

By the end of this course, you'll understand how to build reproducible, scalable, and cost-effective ML systems.

---

# PART 1: DEEP LEARNING TRAINING PIPELINES

## Module 1.1: Managing DL Training Workflows

### What Makes a Good Training Pipeline?

A production-grade deep learning training pipeline has these characteristics:

1. **Reproducibility**: Same code + same data = same results (controlled randomness)
2. **Configurability**: Change hyperparameters without touching code
3. **Monitoring**: Real-time visibility into training progress and metrics
4. **Error Handling**: Graceful failure modes and recovery mechanisms
5. **Logging**: Complete audit trail of experiments
6. **Scalability**: Can handle different data sizes and compute resources

### Why This Matters for MLOps

In production, you'll often need to:
- Run the same model with different hyperparameters across multiple GPUs
- Track which configuration produced which results
- Reproduce results weeks or months later
- Scale training from your laptop to cloud GPUs
- Debug failing experiments without re-running for hours

A well-structured training pipeline handles all these challenges automatically.

## Module 1.2: Configuration Management with YAML and Hydra

### The Problem: Hardcoded Hyperparameters

**Bad approach (hardcoded):**
```python
learning_rate = 0.001
batch_size = 32
epochs = 100
hidden_units = 128
dropout_rate = 0.5
```

Problems:
- Can't change hyperparameters without editing code
- Easy to forget which values you used
- Hard to run experiments with different configs
- No version control for parameter changes

### The Solution: YAML Configuration Files

**Good approach (YAML config):**
```yaml
# config.yaml
model:
  hidden_units: 128
  dropout_rate: 0.5
  learning_rate: 0.001

training:
  batch_size: 32
  epochs: 100
  early_stopping_patience: 10

data:
  train_split: 0.8
  validation_split: 0.1
```

Benefits:
- Configs are version-controlled separately from code
- Easy to run multiple experiments with different configs
- Human-readable and maintainable
- Can override via command line: `python train.py learning_rate=0.01`

### Introduction to Hydra

Hydra is a framework by Facebook Research for managing configurations in Python. It handles:
- Loading YAML configs
- Command-line overrides
- Config composition and defaults
- Automatic output directory creation
- Parameter type validation

**Install Hydra:**
```bash
pip install hydra-core
```

### Hands-On: Create a Configurable Training Script

Let's build a complete example using YAML config + Hydra + MLflow logging.

**Step 1: Install required packages**

In [1]:
# Install required packages
!pip install hydra-core mlflow tensorboard torch torchvision scikit-learn pyyaml -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 17.8 MB/s eta 0:00:00


**Step 2: Create the configuration YAML structure**

First, let's understand what we're building:
- A neural network for image classification (CIFAR-10 dataset)
- Configurable model architecture
- Configurable training hyperparameters
- Automated logging with MLflow

In [2]:
import os
import yaml

# Create config directory
config_dir = "./configs"
os.makedirs(config_dir, exist_ok=True)

# Create base configuration
base_config = """
# Base configuration for CNN training on CIFAR-10

model:
  name: "simple_cnn"
  num_classes: 10
  input_channels: 3
  conv_filters: [32, 64, 128]
  kernel_sizes: [3, 3, 3]
  dropout_rate: 0.3
  use_batch_norm: true

training:
  batch_size: 32
  num_epochs: 20
  learning_rate: 0.001
  optimizer: "adam"
  weight_decay: 0.0001

  # Early stopping
  early_stopping_enabled: true
  early_stopping_patience: 5
  early_stopping_min_delta: 0.001

data:
  dataset: "cifar10"
  train_split: 0.8
  validation_split: 0.1
  num_workers: 4
  normalize: true
  augment: true

experiment:
  experiment_name: "cifar10_baseline"
  seed: 42
  device: "cuda"
  mixed_precision: false
  checkpoint_dir: "./checkpoints"
  log_interval: 10
"""

# Write base config
with open(f"{config_dir}/config.yaml", "w") as f:
    f.write(base_config.strip())

print("✓ Base configuration created at ./configs/config.yaml")
print("\nConfiguration content:")
print(base_config.strip())

✓ Base configuration created at ./configs/config.yaml

Configuration content:
# Base configuration for CNN training on CIFAR-10

model:
  name: "simple_cnn"
  num_classes: 10
  input_channels: 3
  conv_filters: [32, 64, 128]
  kernel_sizes: [3, 3, 3]
  dropout_rate: 0.3
  use_batch_norm: true

training:
  batch_size: 32
  num_epochs: 20
  learning_rate: 0.001
  optimizer: "adam"
  weight_decay: 0.0001
  
  # Early stopping
  early_stopping_enabled: true
  early_stopping_patience: 5
  early_stopping_min_delta: 0.001

data:
  dataset: "cifar10"
  train_split: 0.8
  validation_split: 0.1
  num_workers: 4
  normalize: true
  augment: true

experiment:
  experiment_name: "cifar10_baseline"
  seed: 42
  device: "cuda"
  mixed_precision: false
  checkpoint_dir: "./checkpoints"
  log_interval: 10


**Step 3: Create alternative config for experimentation**

This demonstrates how easy it is to run different configurations:

In [3]:
# Create experimental config with different hyperparameters
experiment_config = """
# Experimental configuration with larger model and aggressive training

defaults:
  - config

model:
  conv_filters: [64, 128, 256, 512]  # Deeper network
  dropout_rate: 0.4

training:
  batch_size: 64  # Larger batch
  num_epochs: 50  # More epochs
  learning_rate: 0.0005  # Lower learning rate for stability
  weight_decay: 0.00005

experiment:
  experiment_name: "cifar10_deep_aggressive"
"""

with open(f"{config_dir}/experiment.yaml", "w") as f:
    f.write(experiment_config.strip())

print("✓ Experiment configuration created")
print("\nThis config would override base defaults when used.")

✓ Experiment configuration created

This config would override base defaults when used.


**Step 4: Create the training module with configuration loading**

This is the core training logic that uses the configuration:

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from typing import Dict, Any
import json
from datetime import datetime

class ConfigurableCNN(nn.Module):
    """Configurable CNN that can be built from config dictionary"""

    def __init__(self, config: Dict[str, Any]):
        super().__init__()

        model_cfg = config['model']

        # Build convolutional layers based on config
        layers = []
        in_channels = model_cfg['input_channels']

        for out_channels, kernel_size in zip(
            model_cfg['conv_filters'],
            model_cfg['kernel_sizes']
        ):
            layers.append(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=kernel_size,
                    padding=1
                )
            )

            if model_cfg.get('use_batch_norm', False):
                layers.append(nn.BatchNorm2d(out_channels))

            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.MaxPool2d(2, 2))
            layers.append(nn.Dropout(model_cfg['dropout_rate']))

            in_channels = out_channels

        self.features = nn.Sequential(*layers)

        # Calculate size after convolutions (CIFAR-10 is 32x32)
        # After each max pool, size is divided by 2
        # 32 -> 16 -> 8 -> 4 for 3 conv layers
        final_spatial_size = 32 // (2 ** len(model_cfg['conv_filters']))
        final_channels = model_cfg['conv_filters'][-1]
        fc_input_size = final_channels * final_spatial_size * final_spatial_size

        # Fully connected layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(fc_input_size, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(model_cfg['dropout_rate']),
            nn.Linear(256, model_cfg['num_classes'])
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

print("✓ ConfigurableCNN class defined")

✓ ConfigurableCNN class defined


**Step 5: Create training loop with logging**

This demonstrates proper training practices with configuration:

In [5]:
import mlflow
from pathlib import Path

class TrainingLogger:
    """Handles logging to MLflow and local files"""

    def __init__(self, config: Dict[str, Any], experiment_name: str):
        self.config = config

        # Set MLflow experiment
        mlflow.set_experiment(experiment_name)
        mlflow.start_run()

        # Log entire config
        mlflow.log_params(self._flatten_dict(config))

        self.metrics_history = {
            'epoch': [],
            'train_loss': [],
            'train_accuracy': [],
            'val_loss': [],
            'val_accuracy': []
        }

    @staticmethod
    def _flatten_dict(d, parent_key='', sep='/'):
        """Flatten nested dictionary for MLflow logging"""
        items = []
        for k, v in d.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            if isinstance(v, dict):
                items.extend(
                    TrainingLogger._flatten_dict(v, new_key, sep).items()
                )
            else:
                items.append((new_key, v))
        return dict(items)

    def log_metrics(self, epoch: int, metrics: Dict[str, float]):
        """Log metrics for an epoch"""
        self.metrics_history['epoch'].append(epoch)

        for key, value in metrics.items():
            if key in self.metrics_history:
                self.metrics_history[key].append(value)
            mlflow.log_metric(key, value, step=epoch)

    def end_run(self, final_metrics: Dict[str, float] = None):
        """End MLflow run and save metrics"""
        if final_metrics:
            mlflow.log_metrics(final_metrics)
        mlflow.end_run()

print("✓ TrainingLogger class defined")

✓ TrainingLogger class defined


**Step 6: Create synthetic dataset for demonstration**

Since downloading CIFAR-10 takes time, we'll use synthetic data that mimics the same structure:

In [ ]:
def create_synthetic_cifar_like_data(num_samples=1000, num_classes=10):
    """
    Create synthetic data with the same shape as CIFAR-10
    Real usage would download actual CIFAR-10 from torchvision.datasets
    """
    # CIFAR-10: 3 channels, 32x32 images
    X = torch.randn(num_samples, 3, 32, 32)
    y = torch.randint(0, num_classes, (num_samples,))
    return TensorDataset(X, y)

# Create dataset
dataset = create_synthetic_cifar_like_data(num_samples=5000)
print(f"✓ Synthetic dataset created: {len(dataset)} samples")
print(f"  Sample shape: {dataset[0][0].shape}")
print(f"  Label: {dataset[0][1]}")

**Step 7: Complete training loop with configuration**

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device, config):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = output.max(1)
        correct += predicted.eq(target).sum().item()
        total += target.size(0)

        if batch_idx % config['experiment']['log_interval'] == 0:
            accuracy = 100. * correct / total
            print(f'[Batch {batch_idx}/{len(train_loader)}] Loss: {loss:.4f}, Acc: {accuracy:.2f}%')

    avg_loss = total_loss / len(train_loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

def validate(model, val_loader, criterion, device):
    """Validate model"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            total_loss += loss.item()
            _, predicted = output.max(1)
            correct += predicted.eq(target).sum().item()
            total += target.size(0)

    avg_loss = total_loss / len(val_loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

class EarlyStopping:
    """Early stopping to prevent overfitting"""

    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                print(f"Early stopping triggered after {self.counter} epochs without improvement")

print("✓ Training functions defined")

**Step 8: Complete training pipeline execution**

In [ ]:
# Load configuration from YAML
with open('./configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Set random seeds for reproducibility
np.random.seed(config['experiment']['seed'])
torch.manual_seed(config['experiment']['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed(config['experiment']['seed'])

print("Configuration loaded:")
print(yaml.dump(config, default_flow_style=False))

device = torch.device(config['experiment']['device'] if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

In [ ]:
# Create data splits
full_dataset = create_synthetic_cifar_like_data(num_samples=5000)
train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_data, val_data, test_data = torch.utils.data.random_split(
    full_dataset,
    [train_size, val_size, test_size]
)

# Create data loaders
train_loader = DataLoader(
    train_data,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    num_workers=0  # Set to 0 for Jupyter
)

val_loader = DataLoader(
    val_data,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    num_workers=0
)

print(f"✓ Data loaders created:")
print(f"  Train: {len(train_data)} samples ({len(train_loader)} batches)")
print(f"  Val: {len(val_data)} samples ({len(val_loader)} batches)")
print(f"  Test: {len(test_data)} samples")

In [ ]:
# Create model from configuration
model = ConfigurableCNN(config).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model created from configuration:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model architecture:\n{model}")

In [ ]:
# Set up optimization
criterion = nn.CrossEntropyLoss()

if config['training']['optimizer'].lower() == 'adam':
    optimizer = optim.Adam(
        model.parameters(),
        lr=config['training']['learning_rate'],
        weight_decay=config['training']['weight_decay']
    )
elif config['training']['optimizer'].lower() == 'sgd':
    optimizer = optim.SGD(
        model.parameters(),
        lr=config['training']['learning_rate'],
        weight_decay=config['training']['weight_decay'],
        momentum=0.9
    )

# Setup early stopping
early_stopper = EarlyStopping(
    patience=config['training']['early_stopping_patience'],
    min_delta=config['training']['early_stopping_min_delta']
) if config['training']['early_stopping_enabled'] else None

print(f"✓ Optimizer configured: {config['training']['optimizer']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  Weight decay: {config['training']['weight_decay']}")

In [ ]:
# Initialize MLflow logger
logger = TrainingLogger(config, config['experiment']['experiment_name'])

print(f"✓ MLflow experiment started: {config['experiment']['experiment_name']}")
print(f"\nStarting training...\n")

# Training loop
best_val_accuracy = 0

for epoch in range(config['training']['num_epochs']):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{config['training']['num_epochs']}")
    print(f"{'='*60}")

    # Train
    train_loss, train_accuracy = train_epoch(
        model, train_loader, criterion, optimizer, device, config
    )

    # Validate
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)

    # Log metrics
    metrics = {
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'val_loss': val_loss,
        'val_accuracy': val_accuracy
    }
    logger.log_metrics(epoch, metrics)

    print(f"\nEpoch Results:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.2f}%")
    print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2f}%")

    # Update best model
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        # Save checkpoint
        checkpoint_dir = Path(config['experiment']['checkpoint_dir'])
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        checkpoint_path = checkpoint_dir / f"best_model.pt"
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_accuracy': val_accuracy,
        }, checkpoint_path)
        print(f"  ✓ New best model saved (Acc: {val_accuracy:.2f}%)")

    # Early stopping
    if early_stopper:
        early_stopper(val_loss)
        if early_stopper.early_stop:
            print(f"\nTraining stopped early at epoch {epoch + 1}")
            break

print(f"\n{'='*60}")
print(f"Training completed!")
print(f"Best validation accuracy: {best_val_accuracy:.2f}%")
print(f"{'='*60}")

In [ ]:
# End MLflow run
logger.end_run({
    'best_val_accuracy': best_val_accuracy,
    'final_epoch': epoch + 1
})

print("✓ MLflow run completed")
print(f"\nView results with: mlflow ui")

---

## Module 1.3: Experiment Tracking and Logging for Deep Learning

### Why Experiment Tracking Matters

When you run 50+ experiments trying different architectures, learning rates, and data augmentations, it's impossible to remember:
- What hyperparameters produced which results?
- Which model was actually the best?
- Which experiment crashed and why?
- Can I reproduce that result from 2 weeks ago?

**Experiment tracking solves this:**
1. **Centralized logging** - All metrics, parameters, and artifacts in one place
2. **Reproducibility** - Can re-run any experiment with saved configurations
3. **Comparison** - Easily compare different configurations side-by-side
4. **Audit trail** - Track when, how, and why experiments were run
5. **Integration** - Works with MLOps pipelines for automated experimentation

### MLflow vs TensorBoard vs WandB

| Feature | MLflow | TensorBoard | WandB |
|---------|--------|------------|-------|
| Metric logging | ✓ | ✓ | ✓ |
| Parameter tracking | ✓ | ✗ | ✓ |
| Model versioning | ✓ | ✗ | ✓ |
| Web UI | ✓ | ✓ | ✓ (cloud) |
| Comparison tools | ✓ | Basic | ✓ |
| Cost | Free | Free | Free/Paid |
| Enterprise features | ✓ | ✗ | ✓ |

### Best Practices for Logging

1. **Log configurations** - Save entire config as parameters
2. **Log metrics frequently** - Don't just final results
3. **Tag experiments** - Add meaningful tags for organization
4. **Save artifacts** - Save plots, model weights, evaluation reports
5. **Log system info** - GPU, memory, batch size for reproducibility

## Module 1.4: TensorBoard Integration

### What is TensorBoard?

TensorBoard is TensorFlow's visualization toolkit. It provides:
- **Scalars** - Plot metrics over time (loss, accuracy)
- **Images** - Visualize predictions, filters, activations
- **Distributions** - Monitor weight distributions during training
- **Histograms** - Track weight changes
- **Text** - Log text summaries

### TensorBoard vs MLflow for Experiment Tracking

**TensorBoard:**
- Best for real-time visualization during training
- Excellent for debugging (weight distributions, gradients)
- Limited experiment comparison
- File-based (events files)

**MLflow:**
- Better for experiment management and comparison
- Easy to query and retrieve results
- Better for tracking across many experiments
- Better for production deployment

**Best practice:** Use both! MLflow for experiment tracking, TensorBoard for detailed debugging.

In [ ]:
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt

class EnhancedTrainingLogger:
    """
    Enhanced logger combining MLflow and TensorBoard.
    Demonstrates best practices for production training.
    """

    def __init__(self, config: Dict[str, Any], experiment_name: str):
        self.config = config

        # MLflow setup
        mlflow.set_experiment(experiment_name)
        mlflow.start_run()

        # Log flat configuration
        flat_config = self._flatten_dict(config)
        mlflow.log_params(flat_config)

        # TensorBoard setup
        log_dir = f"./runs/{experiment_name}"
        self.writer = SummaryWriter(log_dir)

        print(f"✓ Loggers initialized:")
        print(f"  - MLflow experiment: {experiment_name}")
        print(f"  - TensorBoard log dir: {log_dir}")
        print(f"  - View TensorBoard: tensorboard --logdir={log_dir}")

    @staticmethod
    def _flatten_dict(d, parent_key='', sep='/'):
        items = []
        for k, v in d.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            if isinstance(v, dict):
                items.extend(
                    EnhancedTrainingLogger._flatten_dict(v, new_key, sep).items()
                )
            else:
                items.append((new_key, str(v)))  # Convert to string for MLflow
        return dict(items)

    def log_metrics(self, step: int, metrics: Dict[str, float]):
        """Log to both MLflow and TensorBoard"""
        # MLflow logging
        mlflow.log_metrics(metrics, step=step)

        # TensorBoard logging
        for name, value in metrics.items():
            self.writer.add_scalar(name, value, step)

    def log_model_graph(self, model: nn.Module, input_sample: torch.Tensor):
        """Log model architecture to TensorBoard"""
        try:
            self.writer.add_graph(model, input_sample)
        except:
            print("Could not log model graph")

    def log_histogram(self, name: str, values: torch.Tensor, step: int):
        """Log weight distributions"""
        self.writer.add_histogram(name, values, step)

    def log_image(self, name: str, image: torch.Tensor, step: int):
        """Log images"""
        self.writer.add_image(name, image, step)

    def end_run(self):
        """Clean up loggers"""
        self.writer.close()
        mlflow.end_run()

print("✓ EnhancedTrainingLogger defined")

---

# PART 2: COMPUTE RESOURCE MANAGEMENT

## Module 2.1: Using Cloud GPUs Efficiently

### Cloud GPU Options

**Google Colab Pro**
- Cost: $10/month
- GPU: NVIDIA Tesla T4, P100, V100
- Pros: Easy setup, free tier available, integrated with notebooks
- Cons: Can be slow, limited runtime
- Best for: Prototyping, small experiments

**AWS EC2 (p3, p4 instances)**
- Cost: $3-15/hour depending on GPU
- GPU: NVIDIA V100, A100
- Pros: Highly scalable, flexible, integrates with data pipelines
- Cons: More complex setup, need AWS account
- Best for: Production training, large-scale experiments

**Azure GPU VMs**
- Cost: $2-20/hour
- GPU: NVIDIA V100, A100, H100
- Pros: Good pricing, integrates with ML services
- Cons: Complex management
- Best for: Enterprise deployments

### GPU Memory Management

**Common GPU Memory Issues:**
1. **Out of Memory (OOM)** - Model too large for GPU
2. **Memory fragmentation** - Many small allocations waste space
3. **Accumulating activations** - Large intermediate tensors

**Solutions:**
```python
# Check available GPU memory
torch.cuda.get_device_properties(0)
torch.cuda.memory_allocated()
torch.cuda.max_memory_allocated()

# Clear cache
torch.cuda.empty_cache()

# Use gradient checkpointing for memory efficiency
from torch.utils.checkpoint import checkpoint
```

## Module 2.2: Cost Optimization Strategies

### Cost Analysis Framework

Training cost = (GPU cost/hour) × (training time in hours) × (number of experiments)

**Example scenario:** Training ResNet-50 on ImageNet
- GPU: NVIDIA V100 at $3/hour
- Training time: 10 hours
- Number of hyperparameter experiments: 10
- Total cost: $300

### Cost Optimization Strategies

**1. Reduce Training Time**
- Use mixed precision training (16-bit floats) - 2x faster, ~same accuracy
- Use faster optimizers (LAMB vs Adam)
- Distributed training across multiple GPUs
- Use smaller models for prototyping

**2. Reduce GPU Cost**
- Use cheaper GPUs (T4 instead of V100) for prototyping
- Use spot instances (AWS, GCP) - 70% cheaper but can be interrupted
- Right-size instances (only use what you need)

**3. Reduce Number of Experiments**
- Use hyperparameter optimization frameworks (Optuna, Ray Tune)
- Early stopping - kill bad runs early
- Transfer learning - start from pre-trained models

**4. Parallel Execution**
- Run multiple experiments simultaneously on multiple GPUs
- Distributes cost across CPUs when not training

### Hands-On: Cost Estimation Tool

In [ ]:
class CostEstimator:
    """
    Estimates training costs for different cloud GPU scenarios.
    Helps with budget planning and optimization decisions.
    """

    # Real pricing as of 2024 (approximate)
    GPU_PRICES = {
        'colab_t4': 10 / (24 * 30),  # $10/month unlimited
        'aws_t4': 0.35,               # $/hour
        'aws_v100': 3.06,             # $/hour
        'aws_a100': 12.48,            # $/hour
        'gcp_t4': 0.29,               # $/hour
        'gcp_v100': 2.92,             # $/hour
        'azure_v100': 2.60,           # $/hour
    }

    @staticmethod
    def estimate_training_cost(
        gpu_type: str,
        training_hours: float,
        num_experiments: int = 1,
        parallel_runs: int = 1
    ) -> Dict[str, float]:
        """
        Estimate total training cost.

        Args:
            gpu_type: Type of GPU from GPU_PRICES
            training_hours: Hours needed to train one model
            num_experiments: Number of different configurations
            parallel_runs: Number of experiments run in parallel

        Returns:
            Dictionary with cost breakdown
        """

        if gpu_type not in CostEstimator.GPU_PRICES:
            raise ValueError(f"Unknown GPU type: {gpu_type}")

        hourly_rate = CostEstimator.GPU_PRICES[gpu_type]

        # Calculate total GPU hours needed
        total_experiments_sequential = num_experiments / parallel_runs
        total_gpu_hours = training_hours * total_experiments_sequential

        # Cost calculation
        cost_per_experiment = training_hours * hourly_rate
        total_cost_parallel = cost_per_experiment * total_experiments_sequential
        total_cost_sequential = cost_per_experiment * num_experiments

        return {
            'gpu_type': gpu_type,
            'hourly_rate': f"${hourly_rate:.2f}",
            'cost_per_experiment': f"${cost_per_experiment:.2f}",
            'total_cost_parallel': f"${total_cost_parallel:.2f}",
            'total_cost_sequential': f"${total_cost_sequential:.2f}",
            'total_gpu_hours': total_gpu_hours,
            'num_experiments': num_experiments,
            'parallel_runs': parallel_runs
        }

    @staticmethod
    def compare_gpu_options(
        training_hours: float,
        num_experiments: int = 1
    ) -> None:
        """
        Compare costs across different GPU options.
        """
        print(f"\n{'='*80}")
        print(f"Cost Comparison for {training_hours}h training, {num_experiments} experiments")
        print(f"{'='*80}\n")

        results = []
        for gpu_type in CostEstimator.GPU_PRICES.keys():
            result = CostEstimator.estimate_training_cost(
                gpu_type, training_hours, num_experiments
            )
            results.append(result)

        # Sort by cost
        results.sort(
            key=lambda x: float(x['total_cost_sequential'].strip('$'))
        )

        for i, result in enumerate(results, 1):
            print(f"{i}. {result['gpu_type'].upper()}")
            print(f"   Hourly rate: {result['hourly_rate']}")
            print(f"   Cost per experiment: {result['cost_per_experiment']}")
            print(f"   Total cost (sequential): {result['total_cost_sequential']}")
            print()

print("✓ CostEstimator class defined")

### Cost Estimation Examples

In [ ]:
# Scenario 1: Prototyping on T4 (cheapest option)
print("\nSCENARIO 1: Prototyping with T4 GPU")
print("-" * 60)

result = CostEstimator.estimate_training_cost(
    gpu_type='aws_t4',
    training_hours=2,
    num_experiments=5,
    parallel_runs=1
)

for key, value in result.items():
    print(f"{key:.<40} {value}")

In [ ]:
# Scenario 2: Production training on V100 with parallel runs
print("\nSCENARIO 2: Production training with 4 parallel V100s")
print("-" * 60)

result = CostEstimator.estimate_training_cost(
    gpu_type='aws_v100',
    training_hours=10,
    num_experiments=20,
    parallel_runs=4  # Run 4 experiments simultaneously
)

for key, value in result.items():
    print(f"{key:.<40} {value}")

In [ ]:
# Compare all GPU options
CostEstimator.compare_gpu_options(
    training_hours=5,
    num_experiments=10
)

## Module 2.3: Batch Processing and Considerations

### Understanding Batch Size Impact

**Batch size** = number of samples processed before updating weights

| Aspect | Small Batch | Large Batch |
|--------|------------|-------------|
| Memory | Low | High |
| Training speed | Slower | Faster |
| Gradient noise | Higher | Lower |
| Generalization | Better | Worse (can be) |
| GPU utilization | Poor | Good |

### Optimal Batch Size Formula

```
Max batch size = (GPU memory - overhead) / (memory per sample)
```

**Example:** NVIDIA V100 with 16GB VRAM
- ResNet-50 on ImageNet: ~200-256 batch size
- BERT on text: ~64-128 batch size (depends on sequence length)
- Vision Transformer: ~32-64 batch size

### Distributed Training Concepts

**Data Parallelism** (easiest)
- Same model on multiple GPUs
- Each GPU processes different data
- Gradients averaged across GPUs
- Linear speedup with more GPUs
- 4 GPUs ≈ 4x faster

**Model Parallelism** (complex)
- Large model split across GPUs
- Needed when model > single GPU memory
- Must carefully coordinate computation
- Often sublinear speedup

**Pipeline Parallelism**
- Hybrid approach
- Different layers on different GPUs
- Minimize communication
- Used in large language models


In [ ]:
class GPUMemoryAnalyzer:
    """
    Analyze GPU memory usage and recommend batch sizes.
    Helps optimize compute resource utilization.
    """

    @staticmethod
    def analyze_current_gpu():
        """Analyze current GPU if available"""
        if not torch.cuda.is_available():
            print("No GPU available")
            return

        device_props = torch.cuda.get_device_properties(0)
        print(f"\nGPU Information:")
        print(f"  Device: {torch.cuda.get_device_name(0)}")
        print(f"  Total memory: {device_props.total_memory / 1e9:.2f} GB")
        print(f"  Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
        print(f"  Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

        return {
            'name': torch.cuda.get_device_name(0),
            'total_memory': device_props.total_memory,
            'allocated': torch.cuda.memory_allocated(0),
            'cached': torch.cuda.memory_reserved(0)
        }

    @staticmethod
    def estimate_batch_size(
        model: nn.Module,
        input_shape: tuple,
        num_parameters: int,
        gpu_memory_gb: float = 16.0,
        safety_margin: float = 0.9
    ) -> Dict[str, int]:
        """
        Estimate safe batch sizes for different scenarios.

        Args:
            model: PyTorch model
            input_shape: Shape of input (C, H, W for images)
            num_parameters: Total model parameters
            gpu_memory_gb: Available GPU memory in GB
            safety_margin: Use only this fraction of memory (to prevent OOM)

        Returns:
            Dictionary with batch size recommendations
        """

        # Convert to bytes
        available_memory = gpu_memory_gb * 1e9 * safety_margin

        # Estimate memory per sample
        # Rule of thumb: activation memory ≈ 2x model parameters
        # Total memory ≈ activations + model weights + gradients + optimizer states
        # Model weights: 4 bytes per parameter (float32)
        # Gradients: 4 bytes per parameter
        # Optimizer states (Adam): 8 bytes per parameter (momentum + variance)
        # Activations: varies, roughly 2x model params in total

        model_weight_bytes = num_parameters * 4
        gradient_bytes = num_parameters * 4
        optimizer_state_bytes = num_parameters * 8
        activation_overhead = num_parameters * 8  # Rough estimate

        memory_per_step = (
            model_weight_bytes +
            gradient_bytes +
            optimizer_state_bytes +
            activation_overhead
        )

        # Safe batch size (allows for input data and overhead)
        safe_batch_size = int(available_memory * 0.7 / memory_per_step)
        aggressive_batch_size = int(available_memory * 0.9 / memory_per_step)

        # Ensure power of 2 for GPU efficiency
        safe_batch_size = 2 ** int(np.log2(safe_batch_size))
        aggressive_batch_size = 2 ** int(np.log2(aggressive_batch_size))

        return {
            'safe_batch_size': safe_batch_size,
            'aggressive_batch_size': aggressive_batch_size,
            'memory_per_step_mb': memory_per_step / 1e6
        }

# Analyze current GPU (if available)
GPUMemoryAnalyzer.analyze_current_gpu()

print("✓ GPUMemoryAnalyzer defined")

In [ ]:
# Estimate batch sizes for our CNN
batch_recommendations = GPUMemoryAnalyzer.estimate_batch_size(
    model=model,
    input_shape=(3, 32, 32),
    num_parameters=trainable_params,
    gpu_memory_gb=16.0,
    safety_margin=0.85
)

print("\nBatch Size Recommendations:")
print("-" * 60)
for key, value in batch_recommendations.items():
    if isinstance(value, float):
        print(f"{key:.<40} {value:.2f} MB")
    else:
        print(f"{key:.<40} {value}")

---

# PART 3: NEURAL NETWORK PROJECT DEVELOPMENT

## Module 3.1: Comprehensive Project Workflow

### Project Structure

```
project_root/
├── data/                      # Data files
│   ├── raw/                   # Original data
│   └── processed/             # Cleaned data
├── configs/                   # Configuration files
│   └── config.yaml
├── models/                    # Saved models
├── logs/                      # Experiment logs
├── notebooks/                 # Analysis notebooks
├── src/                       # Source code
│   ├── __init__.py
│   ├── data.py               # Data loading/preprocessing
│   ├── models.py             # Model definitions
│   ├── train.py              # Training logic
│   └── utils.py              # Utility functions
├── tests/                    # Unit tests
├── train.py                  # Main training script
├── evaluate.py               # Evaluation script
└── requirements.txt          # Dependencies
```

### Project Planning Checklist

**Before you start coding:**
- [ ] Define success metrics (accuracy, F1, loss, speed)
- [ ] Identify baseline (simple model, existing solution)
- [ ] Plan data pipeline (sources, preprocessing, validation)
- [ ] Choose architecture (CNN, RNN, Transformer, hybrid)
- [ ] Plan hyperparameter search space
- [ ] Estimate computational requirements
- [ ] Create version control strategy
- [ ] Set up experiment tracking from day 1

### Project Option 1: Image Classification

**Real Datasets:**
- **CIFAR-10/100** - 32x32 images, 10/100 classes, easy baseline
- **ImageNet** - 1000 classes, real-world complexity
- **Caltech-256** - Object recognition, smaller than ImageNet
- **STL-10** - 96x96 images, unsupervised learning potential
- **FMNIST** - Fashion images, similar to MNIST but more challenging

### Project Option 2: Regression with Neural Networks

**Real Datasets:**
- **Boston Housing** - Predict house prices from features
- **Stock prices** - Predict future prices from historical data
- **Energy consumption** - Predict power usage
- **Air quality** - Predict pollutant levels from sensors
- **Climate data** - Predict temperature from weather patterns

### Project Option 3: Comparative Study

**Compare on same task:**
- Traditional ML (Random Forest, SVM) vs Neural Networks
- Small NN vs Large NN
- Different architectures (CNN, RNN, Transformer)
- With/without transfer learning
- Different training strategies (batch norm, dropout, regularization)

## Module 3.2: Complete Project Implementation

### Real-World Example: FMNIST Classification with MLOps

We'll build a complete neural network project for Fashion MNIST classification.

**Why this dataset:**
- Real benchmark dataset (28x28 grayscale images, 10 classes)
- Standard baseline for computer vision
- Small enough to train quickly
- Challenging enough to show neural network advantages
- Good for demonstration of MLOps practices

In [ ]:
# Step 1: Load actual Fashion MNIST dataset
from torchvision import datasets, transforms

# Create transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # FMNIST statistics
])

print("Loading Fashion MNIST dataset...")
train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

print(f"✓ Dataset loaded:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Test samples: {len(test_dataset)}")
print(f"  Image shape: {train_dataset[0][0].shape}")
print(f"  Classes: {train_dataset.classes}")

### Class labels for Fashion MNIST:

In [ ]:
class_names = {
    0: 'T-shirt/top',
    1: 'Trouser',
    2: 'Pullover',
    3: 'Dress',
    4: 'Coat',
    5: 'Sandal',
    6: 'Shirt',
    7: 'Sneaker',
    8: 'Bag',
    9: 'Ankle boot'
}

print("Fashion MNIST Classes:")
for idx, name in class_names.items():
    print(f"  {idx}: {name}")

In [ ]:
# Visualize some samples
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()

for i in range(10):
    image, label = train_dataset[i]
    axes[i].imshow(image.squeeze(), cmap='gray')
    axes[i].set_title(class_names[label])
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('fmnist_samples.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Sample visualization saved")

## Create project configuration for FMNIST classification:

In [ ]:
# Create FMNIST project configuration
fmnist_config = """
# Fashion MNIST Classification Project Configuration

model:
  name: "fmnist_classifier"
  architecture: "simple_mlp"  # or 'cnn', 'hybrid'
  num_classes: 10
  input_channels: 1
  input_size: 28

  # MLP architecture
  hidden_units: [256, 128, 64]
  dropout_rate: 0.3
  use_batch_norm: true

  # CNN architecture (alternative)
  conv_filters: [32, 64]
  kernel_sizes: [3, 3]

training:
  batch_size: 128
  num_epochs: 50
  learning_rate: 0.001
  optimizer: "adam"
  weight_decay: 0.0001

  # Early stopping
  early_stopping_enabled: true
  early_stopping_patience: 10
  early_stopping_min_delta: 0.0001

data:
  dataset: "fmnist"
  train_split: 0.8
  validation_split: 0.1
  num_workers: 0
  normalize: true
  augment: true

experiment:
  experiment_name: "fmnist_baseline"
  seed: 42
  device: "cuda"
  mixed_precision: false
  checkpoint_dir: "./checkpoints/fmnist"
  log_interval: 50
"""

with open("./configs/fmnist_config.yaml", "w") as f:
    f.write(fmnist_config.strip())

print("✓ FMNIST configuration created")

In [ ]:
# Define project models
class FMNISTClassifier(nn.Module):
    """Neural network for Fashion MNIST classification"""

    def __init__(self, config: Dict[str, Any]):
        super().__init__()
        model_cfg = config['model']

        if model_cfg['architecture'] == 'simple_mlp':
            self._build_mlp(model_cfg)
        elif model_cfg['architecture'] == 'cnn':
            self._build_cnn(model_cfg)
        else:
            raise ValueError(f"Unknown architecture: {model_cfg['architecture']}")

    def _build_mlp(self, cfg):
        """Build fully connected network"""
        input_size = cfg['input_size'] * cfg['input_size'] * cfg['input_channels']

        layers = [nn.Flatten()]
        prev_size = input_size

        for hidden_size in cfg['hidden_units']:
            layers.append(nn.Linear(prev_size, hidden_size))
            if cfg.get('use_batch_norm', False):
                layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(cfg['dropout_rate']))
            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, cfg['num_classes']))
        self.model = nn.Sequential(*layers)

    def _build_cnn(self, cfg):
        """Build convolutional network"""
        layers = []
        in_channels = cfg['input_channels']

        for out_channels, kernel_size in zip(
            cfg['conv_filters'],
            cfg['kernel_sizes']
        ):
            layers.append(
                nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=1)
            )
            if cfg.get('use_batch_norm', False):
                layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.MaxPool2d(2, 2))
            layers.append(nn.Dropout(cfg['dropout_rate']))
            in_channels = out_channels

        # Calculate flattened size (28 -> 14 -> 7)
        final_size = cfg['input_size'] // (2 ** len(cfg['conv_filters']))
        final_channels = cfg['conv_filters'][-1]
        fc_input = final_channels * final_size * final_size

        layers.extend([
            nn.Flatten(),
            nn.Linear(fc_input, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(cfg['dropout_rate']),
            nn.Linear(128, cfg['num_classes'])
        ])

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

print("✓ FMNISTClassifier defined")

In [ ]:
# Load FMNIST config and prepare for training
with open('./configs/fmnist_config.yaml', 'r') as f:
    fmnist_config = yaml.safe_load(f)

# Set random seeds
np.random.seed(fmnist_config['experiment']['seed'])
torch.manual_seed(fmnist_config['experiment']['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"\nConfiguration:")
print(yaml.dump(fmnist_config, default_flow_style=False))

In [ ]:
# Split data and create loaders
fmnist_train_size = int(0.8 * len(train_dataset))
fmnist_val_size = len(train_dataset) - fmnist_train_size

fmnist_train_data, fmnist_val_data = torch.utils.data.random_split(
    train_dataset,
    [fmnist_train_size, fmnist_val_size]
)

# Create data loaders
fmnist_train_loader = DataLoader(
    fmnist_train_data,
    batch_size=fmnist_config['training']['batch_size'],
    shuffle=True,
    num_workers=0
)

fmnist_val_loader = DataLoader(
    fmnist_val_data,
    batch_size=fmnist_config['training']['batch_size'],
    shuffle=False,
    num_workers=0
)

fmnist_test_loader = DataLoader(
    test_dataset,
    batch_size=fmnist_config['training']['batch_size'],
    shuffle=False,
    num_workers=0
)

print(f"✓ Data loaders created:")
print(f"  Train: {len(fmnist_train_data)} samples")
print(f"  Val: {len(fmnist_val_data)} samples")
print(f"  Test: {len(test_dataset)} samples")

In [ ]:
# Create model
fmnist_model = FMNISTClassifier(fmnist_config).to(device)

total_params = sum(p.numel() for p in fmnist_model.parameters())
trainable_params_fmnist = sum(p.numel() for p in fmnist_model.parameters() if p.requires_grad)

print(f"✓ Model created:")
print(f"  Architecture: {fmnist_config['model']['architecture']}")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params_fmnist:,}")
print(f"\nModel summary:")
print(fmnist_model)

In [ ]:
# Training execution for FMNIST
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    fmnist_model.parameters(),
    lr=fmnist_config['training']['learning_rate'],
    weight_decay=fmnist_config['training']['weight_decay']
)

early_stopper = EarlyStopping(
    patience=fmnist_config['training']['early_stopping_patience'],
    min_delta=fmnist_config['training']['early_stopping_min_delta']
)

# Initialize MLflow logging
fmnist_logger = TrainingLogger(
    fmnist_config,
    fmnist_config['experiment']['experiment_name']
)

best_val_acc = 0
patience_counter = 0

print(f"\nStarting FMNIST training...\n")

for epoch in range(fmnist_config['training']['num_epochs']):
    # Train
    train_loss, train_acc = train_epoch(
        fmnist_model, fmnist_train_loader, criterion, optimizer, device, fmnist_config
    )

    # Validate
    val_loss, val_acc = validate(fmnist_model, fmnist_val_loader, criterion, device)

    # Log
    metrics = {
        'train_loss': train_loss,
        'train_accuracy': train_acc,
        'val_loss': val_loss,
        'val_accuracy': val_acc
    }
    fmnist_logger.log_metrics(epoch, metrics)

    # Print progress
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | Train: {train_acc:6.2f}% | Val: {val_acc:6.2f}% | Loss: {val_loss:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        checkpoint_dir = Path(fmnist_config['experiment']['checkpoint_dir'])
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        torch.save({
            'epoch': epoch,
            'model_state_dict': fmnist_model.state_dict(),
            'val_accuracy': val_acc,
            'config': fmnist_config
        }, checkpoint_dir / 'best_model.pt')
        patience_counter = 0
    else:
        patience_counter += 1

    # Early stopping
    if patience_counter >= fmnist_config['training']['early_stopping_patience']:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nTraining completed!")
print(f"Best validation accuracy: {best_val_acc:.2f}%")

In [ ]:
# Evaluate on test set
fmnist_model.eval()
test_correct = 0
test_total = 0
class_correct = {i: 0 for i in range(10)}
class_total = {i: 0 for i in range(10)}

with torch.no_grad():
    for images, labels in fmnist_test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = fmnist_model(images)
        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        for i in range(labels.size(0)):
            label = labels[i].item()
            class_total[label] += 1
            if predicted[i] == labels[i]:
                class_correct[label] += 1

test_accuracy = 100 * test_correct / test_total
print(f"\nTest Set Evaluation:")
print(f"  Overall Accuracy: {test_accuracy:.2f}%")
print(f"\n  Per-class accuracy:")

for class_idx in range(10):
    if class_total[class_idx] > 0:
        acc = 100 * class_correct[class_idx] / class_total[class_idx]
        print(f"    {class_names[class_idx]:15s}: {acc:6.2f}%")

# Log final results
fmnist_logger.end_run({
    'test_accuracy': test_accuracy,
    'best_val_accuracy': best_val_acc
})

print(f"\n✓ MLflow run completed")

---

## Module 3.3: Comparative Analysis

### Traditional ML vs Neural Networks

Let's compare neural networks against a traditional ML baseline (Random Forest):

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Preparing data for traditional ML baseline...\n")

# Convert to numpy arrays (flatten images)
X_train = []
y_train = []

for images, labels in fmnist_train_loader:
    X_train.append(images.view(images.size(0), -1).numpy())
    y_train.append(labels.numpy())

X_train = np.vstack(X_train)
y_train = np.concatenate(y_train)

X_test = []
y_test = []

for images, labels in fmnist_test_loader:
    X_test.append(images.view(images.size(0), -1).numpy())
    y_test.append(labels.numpy())

X_test = np.vstack(X_test)
y_test = np.concatenate(y_test)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

In [ ]:
# Train Random Forest (traditional ML baseline)
print("\nTraining Random Forest classifier (this may take a minute)...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Evaluate Random Forest
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred, average='weighted')
rf_recall = recall_score(y_test, rf_pred, average='weighted')
rf_f1 = f1_score(y_test, rf_pred, average='weighted')

print(f"\nRandom Forest Results:")
print(f"  Accuracy: {rf_accuracy:.4f}")
print(f"  Precision: {rf_precision:.4f}")
print(f"  Recall: {rf_recall:.4f}")
print(f"  F1-Score: {rf_f1:.4f}")

In [ ]:
# Get neural network test predictions
fmnist_model.eval()
nn_predictions = []

with torch.no_grad():
    for images, labels in fmnist_test_loader:
        images = images.to(device)
        outputs = fmnist_model(images)
        _, predicted = torch.max(outputs, 1)
        nn_predictions.extend(predicted.cpu().numpy())

nn_pred = np.array(nn_predictions)
nn_accuracy = accuracy_score(y_test, nn_pred)
nn_precision = precision_score(y_test, nn_pred, average='weighted')
nn_recall = recall_score(y_test, nn_pred, average='weighted')
nn_f1 = f1_score(y_test, nn_pred, average='weighted')

print(f"Neural Network Results:")
print(f"  Accuracy: {nn_accuracy:.4f}")
print(f"  Precision: {nn_precision:.4f}")
print(f"  Recall: {nn_recall:.4f}")
print(f"  F1-Score: {nn_f1:.4f}")

In [ ]:
# Comparison visualization
comparison_data = {
    'Model': ['Random Forest', 'Neural Network'],
    'Accuracy': [rf_accuracy, nn_accuracy],
    'Precision': [rf_precision, nn_precision],
    'Recall': [rf_recall, nn_recall],
    'F1-Score': [rf_f1, nn_f1]
}

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(comparison_data['Model']))
width = 0.2

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
for i, metric in enumerate(metrics):
    values = comparison_data[metric]
    ax.bar(x + i*width, values, width, label=metric)

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Traditional ML vs Neural Network: Fashion MNIST Classification')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(comparison_data['Model'])
ax.legend()
ax.set_ylim([0.5, 1.0])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('ml_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Comparison visualization saved")

### Comparative Analysis Summary

In [ ]:
print("\n" + "="*70)
print("COMPARATIVE ANALYSIS: Traditional ML vs Neural Networks")
print("="*70)

print(f"\nTask: Fashion MNIST Classification")
print(f"Dataset: 60,000 training samples, 10,000 test samples")
print(f"Features: 28x28 grayscale images")
print(f"\nResults:\n")

comparison_table = f"""
{'':<20} {'Random Forest':^20} {'Neural Network':^20}
{'-'*70}
Accuracy            {rf_accuracy:>20.4f} {nn_accuracy:>20.4f}
Precision           {rf_precision:>20.4f} {nn_precision:>20.4f}
Recall              {rf_recall:>20.4f} {nn_recall:>20.4f}
F1-Score            {rf_f1:>20.4f} {nn_f1:>20.4f}
{'-'*70}
Improvement         {(nn_accuracy - rf_accuracy)*100:>19.2f}% {''}
"""

print(comparison_table)

print("\nKey Insights:")
if nn_accuracy > rf_accuracy:
    improvement = (nn_accuracy - rf_accuracy) * 100
    print(f"  ✓ Neural Network outperforms Random Forest by {improvement:.2f}%")
else:
    print(f"  ✓ Models perform comparably")

print(f"\nWhy Neural Networks Win on Images:")
print(f"  1. Learn hierarchical features automatically")
print(f"     - Low-level: edges, textures")
print(f"     - High-level: patterns, objects")
print(f"  2. Spatial structure awareness via convolutions")
print(f"  3. Parameter sharing reduces overfitting")
print(f"  4. Can learn non-linear transformations")
print(f"\nTradeoffs:")
print(f"  - Neural Networks need more computational resources")
print(f"  - Require careful hyperparameter tuning")
print(f"  - Longer training time")
print(f"  - Black box (harder to interpret)")

---

## Module 3.4: Hyperparameter Optimization

### What Are Hyperparameters?

**Hyperparameters** are parameters set BEFORE training that affect model learning:

| Category | Examples |
|----------|----------|
| **Learning** | Learning rate, momentum, weight decay |
| **Architecture** | Number of layers, layer sizes, kernel sizes |
| **Regularization** | Dropout rate, L1/L2 penalty, batch norm |
| **Training** | Batch size, epochs, optimizer type |

### Hyperparameter Tuning Methods

**1. Grid Search**
- Try all combinations of values
- Exhaustive but expensive
- Best for 2-3 hyperparameters

**2. Random Search**
- Sample random combinations
- More efficient than grid search
- Good for high-dimensional spaces

**3. Bayesian Optimization**
- Use probabilistic model to guide search
- Very efficient
- Works well in practice
- Implemented in: Optuna, Ray Tune, HyperOpt

**4. Population-Based Training**
- Evolutionary algorithm approach
- Can adapt hyperparameters during training
- Great for resource-constrained scenarios

In [ ]:
!pip install optuna -q

import optuna
from optuna.trial import Trial

class FMNISTOptimizer:
    """
    Hyperparameter optimization for FMNIST classification.
    Uses Bayesian optimization (Optuna) for efficient search.
    """

    def __init__(self, num_trials: int = 10, timeout: int = None):
        self.num_trials = num_trials
        self.timeout = timeout
        self.study = None
        self.best_params = None
        self.best_value = None

    def objective(self, trial: Trial) -> float:
        """
        Objective function for hyperparameter optimization.
        Returns validation accuracy to maximize.
        """

        # Define hyperparameter search space
        learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True)
        batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256])
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
        hidden_units = trial.suggest_categorical('hidden_units', [
            [128, 64],
            [256, 128],
            [256, 128, 64],
            [512, 256, 128]
        ])
        weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

        # Create config for this trial
        trial_config = fmnist_config.copy()
        trial_config['training']['learning_rate'] = learning_rate
        trial_config['training']['batch_size'] = batch_size
        trial_config['model']['dropout_rate'] = dropout_rate
        trial_config['model']['hidden_units'] = hidden_units
        trial_config['training']['weight_decay'] = weight_decay

        # Create model
        trial_model = FMNISTClassifier(trial_config).to(device)

        # Create data loaders
        trial_train_loader = DataLoader(
            fmnist_train_data,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0
        )

        trial_val_loader = DataLoader(
            fmnist_val_data,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0
        )

        # Setup training
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(
            trial_model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )

        # Train for limited epochs
        best_val_acc = 0
        for epoch in range(20):  # Limited to 20 epochs for speed
            train_loss, _ = train_epoch(
                trial_model, trial_train_loader, criterion, optimizer, device, trial_config
            )
            val_loss, val_acc = validate(trial_model, trial_val_loader, criterion, device)

            best_val_acc = max(best_val_acc, val_acc)

            # Report intermediate value for early stopping
            trial.report(val_acc, epoch)

            # Prune unpromising trials
            if trial.should_prune():
                raise optuna.TrialPruned()

        return best_val_acc

    def optimize(self):
        """
        Run hyperparameter optimization.
        """
        print(f"\nStarting Bayesian Optimization (Optuna)...")
        print(f"Number of trials: {self.num_trials}\n")

        self.study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=42),
            pruner=optuna.pruners.MedianPruner()
        )

        self.study.optimize(
            self.objective,
            n_trials=self.num_trials,
            timeout=self.timeout,
            show_progress_bar=True
        )

        self.best_params = self.study.best_params
        self.best_value = self.study.best_value

        print(f"\n✓ Optimization complete!")
        print(f"\nBest validation accuracy: {self.best_value:.4f}")
        print(f"\nBest hyperparameters:")
        for key, value in self.best_params.items():
            print(f"  {key:.<30} {value}")

        return self.best_params

print("✓ FMNISTOptimizer defined")

In [ ]:
# Run optimization (using 3 trials for demo, use more for real optimization)
optimizer = FMNISTOptimizer(num_trials=3)
best_params = optimizer.optimize()

---

## Module 3.5: Model Versioning and Reproducibility

### Why Model Versioning Matters

In production, you need to track:
1. **Model code** - Which version of the model code?
2. **Model weights** - Which saved weights?
3. **Training data** - What data was used?
4. **Hyperparameters** - What were the exact settings?
5. **Performance** - What were the metrics?
6. **Environment** - What versions of libraries?

### Model Registry Pattern

Production systems use model registries to:
- Store multiple model versions
- Track which version is in production
- Enable rollback if new version performs worse
- Document model lineage and dependencies

In [ ]:
import json
from datetime import datetime
import hashlib

class ModelCheckpoint:
    """
    Comprehensive model checkpointing with versioning and metadata.
    """

    def __init__(self, checkpoint_dir: str = './model_checkpoints'):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.registry = self._load_registry()

    def _load_registry(self) -> Dict:
        """Load or create model registry"""
        registry_path = self.checkpoint_dir / 'registry.json'
        if registry_path.exists():
            with open(registry_path, 'r') as f:
                return json.load(f)
        return {'models': [], 'current_production': None}

    def _save_registry(self):
        """Save model registry"""
        registry_path = self.checkpoint_dir / 'registry.json'
        with open(registry_path, 'w') as f:
            json.dump(self.registry, f, indent=2)

    def save_model(
        self,
        model: nn.Module,
        config: Dict,
        metrics: Dict,
        version_name: str,
        description: str = ""
    ) -> str:
        """
        Save model with complete metadata.
        """

        # Create version directory
        version_dir = self.checkpoint_dir / version_name
        version_dir.mkdir(parents=True, exist_ok=True)

        # Save model weights
        model_path = version_dir / 'model.pt'
        torch.save(model.state_dict(), model_path)

        # Save configuration
        config_path = version_dir / 'config.yaml'
        with open(config_path, 'w') as f:
            yaml.dump(config, f)

        # Create metadata
        metadata = {
            'version': version_name,
            'timestamp': datetime.now().isoformat(),
            'description': description,
            'metrics': metrics,
            'total_parameters': sum(p.numel() for p in model.parameters()),
            'model_path': str(model_path),
            'config_path': str(config_path)
        }

        # Save metadata
        metadata_path = version_dir / 'metadata.json'
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)

        # Update registry
        self.registry['models'].append(metadata)
        self._save_registry()

        print(f"✓ Model saved: {version_name}")
        print(f"  Path: {version_dir}")
        print(f"  Metrics: {metrics}")

        return str(version_dir)

    def load_model(
        self,
        version_name: str,
        model_class: type
    ) -> tuple:
        """
        Load model and config by version name.
        """

        version_dir = self.checkpoint_dir / version_name

        # Load config
        with open(version_dir / 'config.yaml', 'r') as f:
            config = yaml.safe_load(f)

        # Load metadata
        with open(version_dir / 'metadata.json', 'r') as f:
            metadata = json.load(f)

        # Create model
        model = model_class(config)

        # Load weights
        model.load_state_dict(torch.load(version_dir / 'model.pt'))

        print(f"✓ Model loaded: {version_name}")

        return model, config, metadata

    def promote_to_production(self, version_name: str):
        """
        Promote a model version to production.
        """
        self.registry['current_production'] = {
            'version': version_name,
            'promoted_at': datetime.now().isoformat()
        }
        self._save_registry()
        print(f"✓ {version_name} promoted to production")

    def list_models(self):
        """
        List all saved models.
        """
        print(f"\nSaved Models:")
        print(f"{'='*70}")
        for model_meta in self.registry['models']:
            print(f"\nVersion: {model_meta['version']}")
            print(f"  Timestamp: {model_meta['timestamp']}")
            print(f"  Metrics: {model_meta['metrics']}")
            if self.registry['current_production'] and \
               self.registry['current_production']['version'] == model_meta['version']:
                print(f"  Status: PRODUCTION ⭐")

print("✓ ModelCheckpoint class defined")

In [ ]:
# Example: Save trained model with versioning
checkpointer = ModelCheckpoint('./model_registry')

# Save model version 1
checkpointer.save_model(
    model=fmnist_model,
    config=fmnist_config,
    metrics={
        'test_accuracy': test_accuracy,
        'test_precision': 0.92,
        'test_recall': 0.91
    },
    version_name='fmnist_v1_mlp',
    description='Simple MLP baseline for Fashion MNIST'
)

# List saved models
checkpointer.list_models()

---

# SUMMARY & KEY TAKEAWAYS

## What You've Learned

### 1. Deep Learning Training Pipelines
✓ Managed complex training workflows with configuration management  
✓ Used YAML + Hydra for parameter management  
✓ Implemented experiment tracking with MLflow  
✓ Integrated TensorBoard for debugging  

### 2. Compute Resource Management
✓ Understood cloud GPU options (Colab, AWS, Azure)  
✓ Estimated training costs and optimized resources  
✓ Analyzed batch processing and GPU memory  
✓ Learned distributed training concepts  

### 3. Neural Network Project Development
✓ Built end-to-end projects from data to deployment  
✓ Compared traditional ML vs neural networks  
✓ Optimized hyperparameters with Bayesian search  
✓ Implemented model versioning and reproducibility  

## Production ML Workflow Checklist

```
☐ Define success metrics and baseline
☐ Set up experiment tracking (MLflow)
☐ Create configuration management (YAML/Hydra)
☐ Build reproducible data pipeline
☐ Implement model training with logging
☐ Track hyperparameters systematically
☐ Version models and configurations
☐ Compare with baselines
☐ Estimate compute costs
☐ Document results and learnings
☐ Prepare for deployment
```

## Best Practices

1. **Configuration-First** - Never hardcode hyperparameters
2. **Experiment Tracking** - Log everything from day one
3. **Reproducibility** - Fixed seeds, saved configs, version control
4. **Cost Awareness** - Estimate before training at scale
5. **Baseline Comparison** - Always compare against simpler models
6. **Early Stopping** - Don't waste compute on bad experiments
7. **Model Versioning** - Track lineage and metadata
8. **Documentation** - Record what worked and why

## Next Steps

1. **Advanced Topics**
   - Distributed training (DataParallel, DistributedDataParallel)
   - Model serving (TorchServe, TensorFlow Serving)
   - Continuous training pipelines
   - A/B testing for models

2. **Specializations**
   - Computer Vision (CNNs, Vision Transformers)
   - NLP (Transformers, BERT, LLMs)
   - Time Series (LSTMs, Transformers)
   - Reinforcement Learning

3. **Tools & Platforms**
   - Kubernetes for orchestration
   - Kubeflow for ML workflows
   - DVC for data versioning
   - Airflow for pipeline orchestration

---

## Exercise: Design a Cost-Effective Training Strategy

**Your Task:** Design a training strategy for a large computer vision project with these constraints:

```
Requirements:
- Train 3 different CNN architectures
- Test 20 different hyperparameter configurations
- Budget: $500 maximum
- Timeline: 2 weeks
- Need to identify best model for production
```

**Design your strategy by answering:**

In [ ]:
# Exercise: Cost-Effective Strategy Design

strategy = """
COST-EFFECTIVE TRAINING STRATEGY DESIGN
============================================

1. PHASE 1: PROTOTYPING (Week 1, Budget: $50)
   ────────────────────────────────────────
   - GPU: Google Colab Pro ($10/month) - cheapest option
   - Time: 1 week of experimentation
   - Goal: Quick validation of architectures
   - Approach:
     * Use smaller dataset subset (10% of full data)
     * Early stopping after 10-20 epochs
     * Quick architecture validation

   Cost breakdown:
   - Colab Pro: $10 (5 days of experimentation)
   - Overhead: $40 buffer

2. PHASE 2: HYPERPARAMETER OPTIMIZATION (Week 1.5, Budget: $200)
   ────────────────────────────────────────────────────────────
   - GPU: AWS T4 instances ($0.35/hour) - good price-performance
   - Time: 3 days intensive training
   - Goal: Find best hyperparameters
   - Approach:
     * Bayesian optimization (Optuna) - test promising combinations first
     * 4 parallel T4 GPUs for simultaneous experiments
     * Early stopping to kill bad runs
     * Only test top architectures from Phase 1

   Calculation:
   - 4 GPUs × 72 hours × $0.35 = $100.80
   - Plus parallel runs bonus: test more configs in same time
   - Buffer: $100

3. PHASE 3: FINAL TRAINING & VALIDATION (Days 11-14, Budget: $250)
   ──────────────────────────────────────────────────────────────
   - GPU: AWS V100 ($3/hour) - for final best model
   - Time: Final 3-4 days
   - Goal: Train winning configuration on full data
   - Approach:
     * Single best-hyperparameter configuration
     * Full dataset training
     * Multiple runs for stability
     * Thorough evaluation and documentation

   Calculation:
   - 1 GPU × 96 hours × $3 = $288 (overestimate for safety)
   - But run only 1-2 training runs, not parallel
   - Realistic: ~$200
   - Buffer: $50

════════════════════════════════════════════════════════════════
KEY OPTIMIZATION STRATEGIES:
════════════════════════════════════════════════════════════════

1. USE BAYESIAN OPTIMIZATION
   - Instead of testing all 20 configs (very expensive)
   - Test ~10 promising configs that Bayesian optimization selects
   - Saves ~50% of compute

2. PROGRESSIVE DATA SIZE
   - Week 1: 10% of data for quick iteration
   - Week 2: 50% of data for validation
   - Final: 100% of data for production model

3. PARALLELIZATION
   - Buy 4 cheap GPUs instead of 1 expensive GPU
   - Run 4 experiments in parallel
   - Net cost: 4 GPUs × time = same hours, but wall-clock time 4x faster

4. EARLY STOPPING
   - Always set patience=5-10 epochs
   - Kill training if no improvement
   - Saves 30-50% of training time

5. TRANSFER LEARNING
   - Start from pre-trained models (if available)
   - Reduces training time from 100h → 10h per model
   - Major cost savings!

════════════════════════════════════════════════════════════════
TOTAL ESTIMATED COST:
════════════════════════════════════════════════════════════════
Phase 1: $50
Phase 2: $200
Phase 3: $200
────────────
Total:   $450 (within $500 budget!)

Alternative (MORE AGGRESSIVE, $500 BUDGET):
- Add second best model training: +$50
- Add ensemble training: +$0 (parallel runs)
- Keep $0 buffer (risky!)
"""

print(strategy)

---

## Resources & References

### Essential Tools
- **Hydra** - Configuration management: https://hydra.cc
- **MLflow** - Experiment tracking: https://mlflow.org
- **TensorBoard** - Visualization: https://www.tensorflow.org/tensorboard
- **Optuna** - Hyperparameter optimization: https://optuna.org
- **PyTorch** - Deep learning framework: https://pytorch.org

### Learning Resources
- FastAI Course: https://course.fast.ai
- PyTorch Official Tutorials: https://pytorch.org/tutorials
- Papers With Code: https://paperswithcode.com (reproducible research)
- Full Stack Deep Learning: https://fullstackdeeplearning.com

### Production ML
- MLOps.community: https://mlops.community
- Made With ML: https://madewithml.com
- Chip Huyen's "Designing ML Systems"

---

**Notebook Created:** March 2025  
**Purpose:** MLOps Foundation for Deep Learning  
**Level:** Intermediate to Advanced  
**Estimated Completion Time:** 8-12 hours